# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jh-emon002/flyrank-intern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [25]:
%pip -q install duckdb huggingface_hub scikit-learn

import os
import numpy as np
import pandas as pd
import duckdb
import sklearn

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN not found."

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet'"
    f")"
)

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-15"

EARLY_START = "2026-03-01"
EARLY_END = "2026-03-07"

RECENT_START = "2026-03-09"
RECENT_END = "2026-03-15"

OUTCOME_START = "2026-03-17"
OUTCOME_END = "2026-03-31"

MIN_IMPRESSIONS = 100
DECLINE_THRESHOLD = 0.80

print("sklearn:", sklearn.__version__)
print("Setup complete.")

sklearn: 1.6.1
Setup complete.


In [26]:
df = con.sql(f"""
WITH page_windows AS (

    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN report_date
        END) AS feature_days_available,

        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{OUTCOME_START}'
                                 AND DATE '{OUTCOME_END}'
             AND gsc_data_available IS TRUE
            THEN report_date
        END) AS outcome_days_available,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_pre15,

        -- Early 7 days
        SUM(CASE
            WHEN report_date BETWEEN DATE '{EARLY_START}'
                                 AND DATE '{EARLY_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_early7,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{EARLY_START}'
                                 AND DATE '{EARLY_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_clicks
            ELSE 0
        END) AS clicks_early7,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{EARLY_START}'
                                 AND DATE '{EARLY_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_avg_position * gsc_impressions
            ELSE 0
        END)
        /
        NULLIF(
            SUM(CASE
                WHEN report_date BETWEEN DATE '{EARLY_START}'
                                     AND DATE '{EARLY_END}'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END),
            0
        ) AS avg_position_early7,

        -- Recent 7 days
        SUM(CASE
            WHEN report_date BETWEEN DATE '{RECENT_START}'
                                 AND DATE '{RECENT_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_recent7,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{RECENT_START}'
                                 AND DATE '{RECENT_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_clicks
            ELSE 0
        END) AS clicks_recent7,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{RECENT_START}'
                                 AND DATE '{RECENT_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_avg_position * gsc_impressions
            ELSE 0
        END)
        /
        NULLIF(
            SUM(CASE
                WHEN report_date BETWEEN DATE '{RECENT_START}'
                                     AND DATE '{RECENT_END}'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END),
            0
        ) AS avg_position_recent7,

        -- FUTURE: label construction only
        SUM(CASE
            WHEN report_date BETWEEN DATE '{OUTCOME_START}'
                                 AND DATE '{OUTCOME_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_next15

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM page_windows
WHERE
    feature_days_available = 15
    AND outcome_days_available = 15
    AND impressions_pre15 >= {MIN_IMPRESSIONS}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [27]:
df["recent_trend_pct"] = (
    100.0
    * (df["impressions_recent7"] - df["impressions_early7"])
    / df["impressions_early7"].replace(0, np.nan)
)

df["ctr_early7_pct"] = (
    100.0
    * df["clicks_early7"]
    / df["impressions_early7"].replace(0, np.nan)
)

df["ctr_recent7_pct"] = (
    100.0
    * df["clicks_recent7"]
    / df["impressions_recent7"].replace(0, np.nan)
)

# Positive = ranking position became worse
df["position_change"] = (
    df["avg_position_recent7"]
    - df["avg_position_early7"]
)

df["log_impressions_early7"] = np.log1p(
    df["impressions_early7"]
)

df["log_impressions_recent7"] = np.log1p(
    df["impressions_recent7"]
)

# FUTURE LABEL
df["decline_ratio"] = (
    df["impressions_next15"]
    / df["impressions_pre15"]
)

df["is_declining_next15d"] = (
    df["decline_ratio"] < DECLINE_THRESHOLD
).astype(int)

df = df.replace([np.inf, -np.inf], np.nan)

print("Eligible pages:", len(df))
print(
    "Overall decline base rate:",
    round(df["is_declining_next15d"].mean(), 3)
)

Eligible pages: 58097
Overall decline base rate: 0.345


In [28]:
TARGET = "is_declining_next15d"

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df[TARGET],
        groups=df["client_hash_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df["client_hash_id"])
test_clients = set(test_df["client_hash_id"])

assert train_clients.isdisjoint(test_clients)

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", len(train_clients))
print("Test clients:", len(test_clients))

print(
    "Train base rate:",
    round(train_df[TARGET].mean(), 3)
)

print(
    "Test base rate:",
    round(test_df[TARGET].mean(), 3)
)

print(
    "Client overlap:",
    len(train_clients & test_clients)
)

Train rows: 14350
Test rows: 43747
Train clients: 26
Test clients: 7
Train base rate: 0.296
Test base rate: 0.361
Client overlap: 0


## 1. Two paper findings + my methodology questions

### Finding 1 — The Freshness Multiplier

**Paper finding:** The paper reports that mature 365+ day content that had been
refreshed within 30 days showed approximately 3.2× higher health score and 57×
more impressions than older content in the comparison group.

**Methodology question:** How comparable were the refreshed and unrefreshed
mature-page groups before the refresh occurred?

In particular, I would want to know whether the performance window occurs
strictly after the refresh event and whether pages selected for refreshing
already differed in historical visibility, topic demand, content quality, or
editorial priority. If stronger pages were more likely to be selected for
refresh, some of the measured difference could reflect selection effects rather
than the refresh itself.

The finding is therefore useful as an observed portfolio association, but I
would be cautious about interpreting the 3.2× and 57× differences as causal
effects without a matched, longitudinal, or experimental comparison.

### Finding 2 — Logistic Regression: What Predicts Growth?

**Paper finding:** The exploratory ML appendix reports 71% holdout accuracy for
a Logistic Regression model separating growing from declining pages. Content
age is described as the strongest negative signal, while days visible and
recent impressions are among the stronger positive signals.

**Methodology question:** How exactly was the growing-versus-declining label
constructed, and were all model features measured strictly before the period
used to define that label?

The paper defines trend direction using a 30-day-versus-previous-30-day
impression comparison. I would therefore check whether features such as recent
impressions or days visible overlap either window used to construct the target.

I would also ask whether the reported 80/20 holdout was page-random,
client-grouped, or time-aware. If pages from the same brand occur in both train
and test sets, the model may benefit from shared client-specific characteristics
and the measured performance may overstate generalization to an unseen client.

Finally, I would report the positive-class base rate next to the 71% accuracy,
because accuracy alone does not show how much improvement exists over a simple
majority-class prediction.

In [29]:
import pandas as pd
from IPython.display import display

In [30]:
paper_audit = pd.DataFrame([
    {
        "finding": "Freshness Multiplier",
        "paper_page": 9,
        "reported_result": "3.2x health; 57x impressions",
        "main_audit_question": "Are refreshed and unrefreshed mature pages comparable?"
    },
    {
        "finding": "What Predicts Growth?",
        "paper_page": 29,
        "reported_result": "71% Logistic Regression holdout accuracy",
        "main_audit_question": "Is the label temporally separated and is validation grouped?"
    }
])

display(paper_audit)


,finding,paper_page,reported_result,main_audit_question
0,Freshness Multiplier,9,3.2x health; 57x impressions,Are refreshed and unrefreshed mature pages com...
1,What Predicts Growth?,29,71% Logistic Regression holdout accuracy,Is the label temporally separated and is valid...


## 2. My model under an honest split (before/after)

My Week-5 submission already used a client-grouped holdout. To make the effect
of validation design visible in this audit, I reconstruct a weaker page-random
80/20 split as the "before" condition and compare it with the client-grouped
80/20 split used in Week 5.

The model specification and feature set are held constant. Only the validation
design changes.

The random split allows pages from the same client to occur in both training
and test data. The grouped split assigns each client entirely to one side,
which better measures generalization to clients the model has not seen.

In [31]:
FEATURES = [
    "log_impressions_early7",
    "log_impressions_recent7",
    "recent_trend_pct",
    "ctr_early7_pct",
    "ctr_recent7_pct",
    "avg_position_early7",
    "avg_position_recent7",
    "position_change"
]

TARGET = "is_declining_next15d"


In [32]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

def make_logreg():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                random_state=42
            )
        )
    ])

In [33]:
import numpy as np

In [34]:
from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)

row_ids = np.arange(len(df))

random_train_idx, random_test_idx = train_test_split(
    row_ids,
    test_size=0.20,
    random_state=42,
    stratify=df[TARGET]
)

random_train = df.iloc[random_train_idx].copy()
random_test = df.iloc[random_test_idx].copy()

random_client_overlap = len(
    set(random_train["client_hash_id"])
    &
    set(random_test["client_hash_id"])
)

print("Random train rows:", len(random_train))
print("Random test rows:", len(random_test))
print("Random test base rate:", round(random_test[TARGET].mean(), 3))
print("Clients present in BOTH sets:", random_client_overlap)

Random train rows: 46477
Random test rows: 11620
Random test base rate: 0.345
Clients present in BOTH sets: 28


In [35]:
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

group_train_idx, group_test_idx = next(
    group_splitter.split(
        df,
        y=df[TARGET],
        groups=df["client_hash_id"]
    )
)

group_train = df.iloc[group_train_idx].copy()
group_test = df.iloc[group_test_idx].copy()

group_client_overlap = len(
    set(group_train["client_hash_id"])
    &
    set(group_test["client_hash_id"])
)

print("Grouped train rows:", len(group_train))
print("Grouped test rows:", len(group_test))
print("Grouped train base rate:", round(group_train[TARGET].mean(), 3))
print("Grouped test base rate:", round(group_test[TARGET].mean(), 3))
print("Client overlap:", group_client_overlap)

assert group_client_overlap == 0

Grouped train rows: 14350
Grouped test rows: 43747
Grouped train base rate: 0.296
Grouped test base rate: 0.361
Client overlap: 0


In [36]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

def precision_at_k(y_true, scores, k):

    temp = pd.DataFrame({
        "y": np.asarray(y_true),
        "score": np.asarray(scores)
    })

    temp = temp.sort_values(
        "score",
        ascending=False
    )

    k = min(k, len(temp))

    return temp.head(k)["y"].mean()


def evaluate_scores(y_true, scores):

    return {
        "roc_auc": roc_auc_score(y_true, scores),
        "average_precision": average_precision_score(
            y_true,
            scores
        ),
        "precision_at_10": precision_at_k(
            y_true, scores, 10
        ),
        "precision_at_20": precision_at_k(
            y_true, scores, 20
        ),
        "precision_at_50": precision_at_k(
            y_true, scores, 50
        ),
        "precision_at_100": precision_at_k(
            y_true, scores, 100
        )
    }

In [37]:
random_model = make_logreg()

random_model.fit(
    random_train[FEATURES],
    random_train[TARGET]
)

random_scores = random_model.predict_proba(
    random_test[FEATURES]
)[:, 1]

random_metrics = evaluate_scores(
    random_test[TARGET],
    random_scores
)

In [38]:
grouped_model = make_logreg()

grouped_model.fit(
    group_train[FEATURES],
    group_train[TARGET]
)

grouped_scores = grouped_model.predict_proba(
    group_test[FEATURES]
)[:, 1]

grouped_metrics = evaluate_scores(
    group_test[TARGET],
    grouped_scores
)

In [39]:
split_comparison = pd.DataFrame([
    {
        "validation": "Random page split",
        "test_base_rate": random_test[TARGET].mean(),
        "client_overlap": random_client_overlap,
        **random_metrics
    },
    {
        "validation": "Grouped by client",
        "test_base_rate": group_test[TARGET].mean(),
        "client_overlap": group_client_overlap,
        **grouped_metrics
    }
])

metric_cols = [
    "test_base_rate",
    "roc_auc",
    "average_precision",
    "precision_at_10",
    "precision_at_20",
    "precision_at_50",
    "precision_at_100"
]

split_comparison[metric_cols] = (
    split_comparison[metric_cols].round(3)
)

display(split_comparison)

,validation,test_base_rate,client_overlap,roc_auc,average_precision,precision_at_10,precision_at_20,precision_at_50,precision_at_100
0,Random page split,0.345,28,0.661,0.508,0.9,0.75,0.80,0.77
1,Grouped by client,0.361,0,0.666,0.531,1.0,0.90,0.88,0.87


## 3. Leakage audit
### Prediction timeline

- Feature window: March 1–15, 2026
- Decision point: March 16, 2026
- Outcome window: March 17–31, 2026

All production model inputs are constructed only from information available
before the decision point.

In [40]:
feature_audit = pd.DataFrame([
    ["log_impressions_early7", "Mar 1-7", "Yes",
     "Pre-decision impression volume"],
    ["log_impressions_recent7", "Mar 9-15", "Yes",
     "Pre-decision impression volume"],
    ["recent_trend_pct", "Mar 1-7 vs Mar 9-15", "Yes",
     "Derived only from pre-decision impressions"],
    ["ctr_early7_pct", "Mar 1-7", "Yes",
     "Pre-decision clicks / impressions"],
    ["ctr_recent7_pct", "Mar 9-15", "Yes",
     "Pre-decision clicks / impressions"],
    ["avg_position_early7", "Mar 1-7", "Yes",
     "Pre-decision weighted search position"],
    ["avg_position_recent7", "Mar 9-15", "Yes",
     "Pre-decision weighted search position"],
    ["position_change", "Mar 1-7 vs Mar 9-15", "Yes",
     "Derived only from pre-decision positions"]
], columns=[
    "feature",
    "source_window",
    "known_at_decision_time",
    "note"
])

display(feature_audit)


,feature,source_window,known_at_decision_time,note
0,log_impressions_early7,Mar 1-7,Yes,Pre-decision impression volume
1,log_impressions_recent7,Mar 9-15,Yes,Pre-decision impression volume
2,recent_trend_pct,Mar 1-7 vs Mar 9-15,Yes,Derived only from pre-decision impressions
3,ctr_early7_pct,Mar 1-7,Yes,Pre-decision clicks / impressions
4,ctr_recent7_pct,Mar 9-15,Yes,Pre-decision clicks / impressions
5,avg_position_early7,Mar 1-7,Yes,Pre-decision weighted search position
6,avg_position_recent7,Mar 9-15,Yes,Pre-decision weighted search position
7,position_change,Mar 1-7 vs Mar 9-15,Yes,Derived only from pre-decision positions


In [41]:
FORBIDDEN = {
    "impressions_next15",
    "decline_ratio",
    "is_declining_next15d",
    "trend_direction",
    "trend_pct",
    "health_score",
    "client_hash_id",
    "content_hash_id"
}

overlap = set(FEATURES) & FORBIDDEN

print("Forbidden feature overlap:", overlap)

assert len(overlap) == 0

print("Direct leakage check: PASS")

Forbidden feature overlap: set()
Direct leakage check: PASS


### Label-definition sensitivity

The two strongest Week-5 predictors are derived from the same pre-decision
impression history that contributes to the denominator of the future-decline
label.

This is not direct future leakage because both variables are known before the
decision point. However, it creates target-definition coupling: the model may
partly exploit the construction of the relative-decline label rather than learn
independent signals of future deterioration.

I therefore run a sensitivity test removing the most directly
impression-derived predictors.

In [42]:
REDUCED_FEATURES = [
    "ctr_early7_pct",
    "ctr_recent7_pct",
    "avg_position_early7",
    "avg_position_recent7",
    "position_change"
]

In [43]:
reduced_model = make_logreg()

reduced_model.fit(
    group_train[REDUCED_FEATURES],
    group_train[TARGET]
)

reduced_scores = reduced_model.predict_proba(
    group_test[REDUCED_FEATURES]
)[:, 1]

reduced_metrics = evaluate_scores(
    group_test[TARGET],
    reduced_scores
)

sensitivity = pd.DataFrame([
    {
        "model": "Full Week-5 features",
        **grouped_metrics
    },
    {
        "model": "Without impression-derived predictors",
        **reduced_metrics
    }
])

sensitivity[
    [
        "roc_auc",
        "average_precision",
        "precision_at_10",
        "precision_at_20",
        "precision_at_50",
        "precision_at_100"
    ]
] = sensitivity[
    [
        "roc_auc",
        "average_precision",
        "precision_at_10",
        "precision_at_20",
        "precision_at_50",
        "precision_at_100"
    ]
].round(3)

display(sensitivity)

,model,roc_auc,average_precision,precision_at_10,precision_at_20,precision_at_50,precision_at_100
0,Full Week-5 features,0.666,0.531,1.0,0.9,0.88,0.87
1,Without impression-derived predictors,0.599,0.423,0.5,0.4,0.36,0.37


The model is strongly dependent on pre-decision impression history, which is also closely related to how the target is defined. Because these inputs are available at decision time, this is not direct temporal leakage, but the sensitivity result narrows the interpretation of the model.

## 4. Claim rewrite

## 4. Claim rewrite

### Claim 1

**Too broad**

> Logistic Regression is the best model for identifying future-declining pages.

**Rewritten**

> On the March client-grouped holdout, Logistic Regression achieved the highest
> measured Precision@K among the Week-5 methods at K = 10, 20, 50, and 100.
> This supports using its probability score as a decision-support ranking in
> this evaluation. It does not establish that the model will outperform the
> alternatives for every unseen client or future time period.

### Claim 2

**Too broad**

> High earlier visibility followed by low recent visibility predicts future
> decline.

**Rewritten**

> Within the fitted Logistic Regression model, higher earlier impression
> exposure combined with lower recent exposure was directionally associated
> with a higher predicted probability of meeting the constructed
> future-decline label. The association is measured within this dataset and
> should not be interpreted as causal.

In [44]:
claim_receipt = pd.DataFrame([
    {
        "evidence": "Grouped holdout base rate",
        "value": round(group_test[TARGET].mean(), 3)
    },
    {
        "evidence": "Grouped ROC-AUC",
        "value": round(grouped_metrics["roc_auc"], 3)
    },
    {
        "evidence": "Grouped average precision",
        "value": round(
            grouped_metrics["average_precision"],
            3
        )
    },
    {
        "evidence": "Grouped Precision@100",
        "value": round(
            grouped_metrics["precision_at_100"],
            3
        )
    }
])

display(claim_receipt)


,evidence,value
0,Grouped holdout base rate,0.361
1,Grouped ROC-AUC,0.666
2,Grouped average precision,0.531
3,Grouped Precision@100,0.870


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.